In [1]:
from __future__ import annotations

import csv
import json
import math
import random
import sys
from copy import deepcopy
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from mionet_linear import LinearMIONet, MIONetConfig, mse_operator_loss

In [2]:
THIS_DIR = Path.cwd()

DEFAULT_CONFIG = {
    "data": {
        "data_dir": "./Data/data/mionet_gp_2d_5000",
        "solution_dir": "./Data/data/mionet_gp_2d_5000/solved_5000",
        "u_path": None,
        "k_sensor_file": "k_sensor.npy",
        "f_sensor_file": "f_sensor.npy",
        "output_points_file": "output_points.npy",
        "u_train_file": "u_train.npy",
        # For quick debugging. Use null/None for all samples.
        "sample_limit": None,
    },
    "model": {
        "latent_dim": 500,
        "k_hidden": [500, 500, 500],
        "trunk_hidden": [500, 500, 500],
        "activation": "tanh",
    },
    "train": {
        "epochs": 2000,
        "batch_size": 8,
        # 0 or negative means use all output points.
        "point_batch_size": 4096,
        "train_ratio": 0.9,
        "lr": 1.0e-4,
        "weight_decay": 0.0,
        "grad_clip": 1.0,
        "eval_every": 10,
        "save_every": 100,
        "num_workers": 0,
        "seed": 12345,
        "resume": None,
        # Validation can be expensive; limit number of validation batches.
        "val_max_batches": 20,
    },
    "runtime": {
        "device": "cuda",
        "dtype": "float32",
        "pin_memory": True,
    },
    "output": {
        "out_dir": "./mionet_ckpt",
        "overwrite_history": True,
    },
}


In [3]:
def deep_update(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    """Recursively update base dict."""
    out = deepcopy(base)
    for k, v in override.items():
        if isinstance(v, dict) and isinstance(out.get(k), dict):
            out[k] = deep_update(out[k], v)
        else:
            out[k] = v
    return out

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


In [6]:
class MIONetNPYDataset(Dataset):
    def __init__(self, config: dict[str, Any], np_dtype: np.dtype):
        data_cfg = config["data"]

        data_dir = Path(data_cfg["data_dir"])
        solution_dir = Path(data_cfg["solution_dir"]) if data_cfg.get("solution_dir") is not None else None

        k_path = data_dir / data_cfg.get("k_sensor_file", "k_sensor.npy")
        f_path = data_dir / data_cfg.get("f_sensor_file", "f_sensor.npy")
        coords_path = data_dir / data_cfg.get("output_points_file", "output_points.npy")

        u_path_cfg = data_cfg.get("u_path", None)
        if u_path_cfg:
            u_path = Path(u_path_cfg)
        else:
            if solution_dir is None:
                raise ValueError("data.solution_dir is required when data.u_path is not provided.")
            u_path = solution_dir / data_cfg.get("u_train_file", "u_train.npy")

        self.k_sensor = np.load(k_path, mmap_mode="r")
        self.f_sensor = np.load(f_path, mmap_mode="r")
        self.output_points = np.load(coords_path)
        self.u_train = np.load(u_path, mmap_mode="r")

        sample_limit = data_cfg.get("sample_limit", None)
        if sample_limit is not None:
            sample_limit = int(sample_limit)
            sample_limit = min(sample_limit, self.k_sensor.shape[0])
            self.indices = np.arange(sample_limit)
        else:
            self.indices = np.arange(self.k_sensor.shape[0])

        self.np_dtype = np_dtype

        self.paths = {
            "k_sensor": str(k_path),
            "f_sensor": str(f_path),
            "output_points": str(coords_path),
            "u_train": str(u_path),
        }

    def __len__(self) -> int:
        return len(self.indices)

    @property
    def k_dim(self) -> int:
        return self.k_sensor.shape[1]

    @property
    def f_dim(self) -> int:
        return self.f_sensor.shape[1]

    @property
    def n_out(self) -> int:
        return self.u_train.shape[1]

    def __getitem__(self, local_idx: int):
        idx = int(self.indices[local_idx])
        k = np.asarray(self.k_sensor[idx], dtype=self.np_dtype)
        f = np.asarray(self.f_sensor[idx], dtype=self.np_dtype)
        u = np.asarray(self.u_train[idx], dtype=self.np_dtype)
        return k, f, u


def make_splits(n: int, train_ratio: float, seed: int) -> tuple[list[int], list[int]]:
    rng = np.random.default_rng(seed)
    idx = np.arange(n)
    rng.shuffle(idx)

    n_train = int(n * train_ratio)
    if n_train <= 0 or n_train >= n:
        raise ValueError(
            f"Invalid train_ratio={train_ratio}. Need both train and val samples. "
            f"n={n}, n_train={n_train}"
        )

    return idx[:n_train].tolist(), idx[n_train:].tolist()


def get_device(config: dict[str, Any]) -> torch.device:
    dev = config["runtime"].get("device", "auto")
    if dev == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return torch.device(dev)


def get_dtypes(config: dict[str, Any]) -> tuple[torch.dtype, np.dtype]:
    dtype = config["runtime"].get("dtype", "float32")
    if dtype == "float64":
        torch.set_default_dtype(torch.float64)
        return torch.float64, np.float64
    if dtype == "float32":
        torch.set_default_dtype(torch.float32)
        return torch.float32, np.float32
    raise ValueError(f"Unsupported dtype: {dtype}")


def get_activation(name: str):
    activation_map = {
        "tanh": nn.Tanh,
        "gelu": nn.GELU,
        "relu": nn.ReLU,
        "silu": nn.SiLU,
    }
    if name not in activation_map:
        raise ValueError(f"Unknown activation={name}. Choose from {list(activation_map)}")
    return activation_map[name]


def to_device_batch(batch, device: torch.device, dtype: torch.dtype):
    k, f, u = batch
    return (
        k.to(device=device, dtype=dtype, non_blocking=True),
        f.to(device=device, dtype=dtype, non_blocking=True),
        u.to(device=device, dtype=dtype, non_blocking=True),
    )


def sample_output_indices(n_out: int, point_batch_size: int, device: torch.device) -> torch.Tensor:
    if point_batch_size is None or point_batch_size <= 0 or point_batch_size >= n_out:
        return torch.arange(n_out, device=device)
    return torch.randperm(n_out, device=device)[:point_batch_size]


def relative_l2(pred: torch.Tensor, target: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    num = torch.linalg.norm(pred - target, dim=1)
    den = torch.linalg.norm(target, dim=1).clamp_min(eps)
    return torch.mean(num / den)


@torch.no_grad()
def evaluate(
    *,
    model: nn.Module,
    loader: DataLoader,
    coords_all: torch.Tensor,
    device: torch.device,
    dtype: torch.dtype,
    point_batch_size: int,
    max_batches: int,
) -> dict[str, float]:
    model.eval()

    mse_sum = 0.0
    rel_sum = 0.0
    count = 0

    n_out = coords_all.shape[0]

    for batch_id, batch in enumerate(loader):
        if batch_id >= max_batches:
            break

        k, f, u = to_device_batch(batch, device, dtype)
        point_idx = sample_output_indices(n_out, point_batch_size, device)

        coords = coords_all[point_idx]
        target = u[:, point_idx]
        pred = model(k, f, coords)

        mse = mse_operator_loss(pred, target)
        rel = relative_l2(pred, target)

        bs = k.shape[0]
        mse_sum += float(mse.item()) * bs
        rel_sum += float(rel.item()) * bs
        count += bs

    if count == 0:
        return {"mse": math.nan, "rel_l2": math.nan}

    return {"mse": mse_sum / count, "rel_l2": rel_sum / count}


def save_checkpoint(
    *,
    path: Path,
    model: nn.Module,
    optimizer: torch.optim.Optimizer,
    epoch: int,
    config: dict[str, Any],
    metrics: dict[str, float],
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "config": config,
            "metrics": metrics,
        },
        path,
    )


def train_from_config(user_config: dict[str, Any] | str | Path) -> dict[str, Any]:
    """Train MIONet from a Python dict or YAML path.

    Returns:
        {
            "model": model,
            "best_path": Path,
            "last_path": Path,
            "history_path": Path,
            "config": config,
        }
    """
    config = deep_update(DEFAULT_CONFIG, user_config)

    train_cfg = config["train"]
    model_cfg = config["model"]
    out_cfg = config["output"]

    set_seed(int(train_cfg.get("seed", 0)))

    device = get_device(config)
    torch_dtype, np_dtype = get_dtypes(config)

    out_dir = Path(out_cfg["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    dataset = MIONetNPYDataset(config, np_dtype=np_dtype)

    print("Dataset:")
    print(f"  k_sensor:      {dataset.k_sensor.shape}")
    print(f"  f_sensor:      {dataset.f_sensor.shape}")
    print(f"  u_train:       {dataset.u_train.shape}")
    print(f"  output_points: {dataset.output_points.shape}")
    print(f"  used samples:  {len(dataset)}")
    print(f"  paths:         {dataset.paths}")
    print("Runtime:")
    print(f"  device:        {device}")
    print(f"  dtype:         {torch_dtype}")

    train_idx, val_idx = make_splits(
        n=len(dataset),
        train_ratio=float(train_cfg.get("train_ratio", 0.9)),
        seed=int(train_cfg.get("seed", 0)),
    )

    pin_memory = bool(config["runtime"].get("pin_memory", True)) and device.type == "cuda"

    train_loader = DataLoader(
        Subset(dataset, train_idx),
        batch_size=int(train_cfg.get("batch_size", 8)),
        shuffle=True,
        num_workers=int(train_cfg.get("num_workers", 0)),
        pin_memory=pin_memory,
        drop_last=False,
    )
    val_loader = DataLoader(
        Subset(dataset, val_idx),
        batch_size=int(train_cfg.get("batch_size", 8)),
        shuffle=False,
        num_workers=int(train_cfg.get("num_workers", 0)),
        pin_memory=pin_memory,
        drop_last=False,
    )

    mcfg = MIONetConfig(
        k_dim=dataset.k_dim,
        f_dim=dataset.f_dim,
        coord_dim=2,
        latent_dim=int(model_cfg.get("latent_dim", 500)),
        k_hidden=tuple(model_cfg.get("k_hidden", [500, 500, 500])),
        trunk_hidden=tuple(model_cfg.get("trunk_hidden", [500, 500, 500])),
        activation=get_activation(model_cfg.get("activation", "tanh")),
    )

    model = LinearMIONet(mcfg).to(device=device, dtype=torch_dtype)
    print("model parameter number", sum(p.numel() for p in model.parameters()))
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=float(train_cfg.get("lr", 1e-4)),
        weight_decay=float(train_cfg.get("weight_decay", 0.0)),
    )

    start_epoch = 1
    best_val_rel = float("inf")

    resume = train_cfg.get("resume", None)
    if resume:
        ckpt = torch.load(resume, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = int(ckpt["epoch"]) + 1
        best_val_rel = float(ckpt.get("metrics", {}).get("best_val_rel_l2", best_val_rel))
        print(f"Resumed from {resume}, start_epoch={start_epoch}")

    coords_all = torch.as_tensor(dataset.output_points, dtype=torch_dtype, device=device)

    # Save fully resolved config.
    with open(out_dir / "resolved_config.json", "w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    history_path = out_dir / "history.csv"
    overwrite_history = bool(out_cfg.get("overwrite_history", True))
    if overwrite_history or not history_path.exists():
        with open(history_path, "w", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow(["epoch", "train_mse", "train_rel_l2", "val_mse", "val_rel_l2", "lr"])

    epochs = int(train_cfg.get("epochs", 2000))
    point_batch_size = int(train_cfg.get("point_batch_size", 4096))
    eval_every = int(train_cfg.get("eval_every", 10))
    save_every = int(train_cfg.get("save_every", 100))
    grad_clip = float(train_cfg.get("grad_clip", 1.0))
    val_max_batches = int(train_cfg.get("val_max_batches", 20))

    print("Start training.")

    for epoch in range(start_epoch, epochs + 1):
        model.train()

        train_mse_sum = 0.0
        train_rel_sum = 0.0
        train_count = 0

        for batch in train_loader:
            k, f, u = to_device_batch(batch, device, torch_dtype)

            point_idx = sample_output_indices(dataset.n_out, point_batch_size, device)
            coords = coords_all[point_idx]
            target = u[:, point_idx]

            pred = model(k, f, coords)

            loss = mse_operator_loss(pred, target)
            rel = relative_l2(pred, target)

            optimizer.zero_grad(set_to_none=True)
            loss.backward()

            if grad_clip > 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)

            optimizer.step()

            bs = k.shape[0]
            train_mse_sum += float(loss.item()) * bs
            train_rel_sum += float(rel.item()) * bs
            train_count += bs

        train_mse = train_mse_sum / max(train_count, 1)
        train_rel = train_rel_sum / max(train_count, 1)

        val_metrics = {"mse": math.nan, "rel_l2": math.nan}

        if epoch == 1 or epoch % eval_every == 0:
            val_metrics = evaluate(
                model=model,
                loader=val_loader,
                coords_all=coords_all,
                device=device,
                dtype=torch_dtype,
                point_batch_size=point_batch_size,
                max_batches=val_max_batches,
            )

            if val_metrics["rel_l2"] < best_val_rel:
                best_val_rel = val_metrics["rel_l2"]
                save_checkpoint(
                    path=out_dir / "best.pt",
                    model=model,
                    optimizer=optimizer,
                    epoch=epoch,
                    config=config,
                    metrics={
                        "train_mse": train_mse,
                        "train_rel_l2": train_rel,
                        "val_mse": val_metrics["mse"],
                        "val_rel_l2": val_metrics["rel_l2"],
                        "best_val_rel_l2": best_val_rel,
                    },
                )

        lr = optimizer.param_groups[0]["lr"]

        with open(history_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f)
            writer.writerow([epoch, train_mse, train_rel, val_metrics["mse"], val_metrics["rel_l2"], lr])

        if epoch == 1 or epoch % eval_every == 0:
            print(
                f"epoch={epoch:05d} "
                f"train_mse={train_mse:.3e} train_rel={train_rel:.3e} "
                f"val_mse={val_metrics['mse']:.3e} val_rel={val_metrics['rel_l2']:.3e} "
                f"best_val_rel={best_val_rel:.3e}"
            )

        if save_every > 0 and epoch % save_every == 0:
            save_checkpoint(
                path=out_dir / f"epoch_{epoch:06d}.pt",
                model=model,
                optimizer=optimizer,
                epoch=epoch,
                config=config,
                metrics={
                    "train_mse": train_mse,
                    "train_rel_l2": train_rel,
                    "val_mse": val_metrics["mse"],
                    "val_rel_l2": val_metrics["rel_l2"],
                    "best_val_rel_l2": best_val_rel,
                },
            )

    save_checkpoint(
        path=out_dir / "last.pt",
        model=model,
        optimizer=optimizer,
        epoch=epochs,
        config=config,
        metrics={"best_val_rel_l2": best_val_rel},
    )

    # Linearity check in f branch
    model.eval()
    with torch.no_grad():
        one_batch = next(iter(val_loader))
        k0, _, _ = to_device_batch(one_batch, device, torch_dtype)
        k0 = k0[:1]
        f1 = torch.randn((1, dataset.f_dim), device=device, dtype=torch_dtype)
        f2 = torch.randn((1, dataset.f_dim), device=device, dtype=torch_dtype)
        coords_test = coords_all[: min(128, coords_all.shape[0])]
        passed, err = model.check_linearity_in_f(k0, f1, f2, coords_test)

    print("Training finished.")
    print(f"Linearity check in f: passed={passed}, max_abs_error={err:.3e}")
    print("Best checkpoint:", out_dir / "best.pt")
    print("Last checkpoint:", out_dir / "last.pt")

    return {
        "model": model,
        "best_path": out_dir / "best.pt",
        "last_path": out_dir / "last.pt",
        "history_path": history_path,
        "config": config,
        "best_val_rel_l2": best_val_rel,
    }

# parser = argparse.ArgumentParser()
# parser.add_argument("--config", type=str, default="train_mionet_2d.yaml")
# args = parser.parse_args()
result = train_from_config(DEFAULT_CONFIG)





Dataset:
  k_sensor:      (5000, 10000)
  f_sensor:      (5000, 10000)
  u_train:       (5000, 10201)
  output_points: (10201, 2)
  used samples:  5000
  paths:         {'k_sensor': 'Data\\data\\mionet_gp_2d_5000\\k_sensor.npy', 'f_sensor': 'Data\\data\\mionet_gp_2d_5000\\f_sensor.npy', 'output_points': 'Data\\data\\mionet_gp_2d_5000\\output_points.npy', 'u_train': 'Data\\data\\mionet_gp_2d_5000\\solved_5000\\u_train.npy'}
Runtime:
  device:        cuda
  dtype:         torch.float32
model parameter number 11505000
Start training.
epoch=00001 train_mse=1.195e-02 train_rel=1.562e+00 val_mse=1.558e-04 val_rel=5.765e-01 best_val_rel=5.765e-01


KeyboardInterrupt: 

NameError: name 'model' is not defined